# N-Gram + BPC Notebook

This notebook is designed for the current Task 2 pipeline:

- unigram / bigram / trigram
- Laplace smoothing
- next-token prediction
- sentence scoring
- **Bits Per Character (BPC)**
- perplexity (PPL)

It is especially useful for `text8`, where `--max-fit-texts` is not very helpful because the training split is stored as a single long text stream. In this notebook, we use `max_fit_characters` instead.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from typing import Optional

REPO_URL = "https://github.com/HatakekkSheeshh/text-preprocess-tokenization.git"
GIT_REF = "bpc-metric"  # change this if you want to checkout another pushed branch
REPO_NAME = "text-preprocess-tokenization"
REPO_DIR = None      # e.g. "/root/text-preprocess-tokenization" if you already mounted/cloned the repo
AUTO_CLONE_IF_MISSING = True
AUTO_PULL_LATEST = False


def looks_like_repo_root(path: Path) -> bool:
    return (path / "requirements.txt").exists() and (path / "src").exists() and (path / "main.py").exists()


def find_repo_root(start: Path) -> Optional[Path]:
    common_roots = [start, *start.parents, Path.home(), Path("/root"), Path("/content"), Path("/workspace"), Path("/mnt"), Path("/tmp")]
    seen = set()
    candidates = []
    for root in common_roots:
        if not root.exists():
            continue
        candidates.append(root)
        candidates.append(root / REPO_NAME)
        try:
            for child in root.iterdir():
                if child.is_dir() and child.name == REPO_NAME:
                    candidates.append(child)
        except OSError:
            pass

    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if looks_like_repo_root(candidate):
            return candidate
    return None


def run_git(*args: str) -> None:
    subprocess.run(["git", *args], check=True)


if REPO_DIR is not None:
    project_root = Path(REPO_DIR).expanduser().resolve()
    if not looks_like_repo_root(project_root):
        raise FileNotFoundError(f"REPO_DIR does not look like the repo root: {project_root}")
else:
    detected_root = find_repo_root(Path.cwd().resolve())
    if detected_root is None and AUTO_CLONE_IF_MISSING:
        clone_parent = Path("/content") if Path("/content").exists() else Path.home()
        project_root = (clone_parent / REPO_NAME).resolve()
        if not project_root.exists():
            run_git("clone", REPO_URL, str(project_root))
        run_git("-C", str(project_root), "checkout", GIT_REF)
    elif detected_root is None:
        raise FileNotFoundError(
            "Could not find the project root automatically. "
            "Set REPO_DIR to the repo path, or allow AUTO_CLONE_IF_MISSING."
        )
    else:
        project_root = detected_root

if AUTO_PULL_LATEST:
    run_git("-C", str(project_root), "fetch", "origin")
    run_git("-C", str(project_root), "checkout", GIT_REF)
    run_git("-C", str(project_root), "pull", "--ff-only", "origin", GIT_REF)

os.chdir(project_root)
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python import root added: {PROJECT_ROOT}")

In [ ]:
!pip install -q -r "{PROJECT_ROOT / 'requirements.txt'}"

In [ ]:
import json
import shutil

import pandas as pd

from src.datasets.load_data import load
from src.training.train_ngram import NGramTrainingConfig, train_ngram_language_model

METRICS_ROOT = PROJECT_ROOT / "outputs" / "metrics" / "ngram"
ARTIFACT_ROOT = PROJECT_ROOT / "outputs" / "artifacts" / "ngram"


def load_metrics(run_name: str) -> dict:
    metrics_path = METRICS_ROOT / f"{run_name}.json"
    return json.loads(metrics_path.read_text(encoding="utf-8"))


def summarize_metrics(metrics: dict) -> pd.DataFrame:
    rows = []
    for split_name in ("train", "validation", "test"):
        split = metrics["splits"][split_name]
        rows.append(
            {
                "split": split_name,
                "num_tokens": split["num_tokens"],
                "num_characters": split["num_characters"],
                "avg_nll": round(split["average_negative_log_likelihood"], 4),
                "bpc": round(split["bits_per_character"], 4),
                "ppl": round(split["perplexity"], 4),
            }
        )
    return pd.DataFrame(rows)


## Run configuration

Recommended defaults for a quick Colab run on `text8`:

- `PROFILE = "quick"`
- `DATASET_NAME = "text8"`
- `TOKENIZER_NAME = "word"` or `"char"`
- `NGRAM_ORDER = 2` or `3`

For `text8` and `enwik8`, the key speed knob is `max_fit_characters`, not `max_fit_texts`.

In [ ]:
DATASET_NAME = "text8"
TOKENIZER_NAME = "word"  # word | char | bpe
NGRAM_ORDER = 2           # 1 | 2 | 3
LAPLACE_ALPHA = 1.0
PROFILE = "quick"        # quick | medium | full

PREDICTION_CONTEXTS = ["the history"]
SCORE_TEXTS = [
    "the history of science",
    "science of history the",
]

PROFILES = {
    "quick": {
        "text8": {"max_fit_texts": None, "max_fit_characters": 200_000, "max_train_tokens": 20_000, "max_validation_tokens": 5_000, "max_test_tokens": 5_000},
        "enwik8": {"max_fit_texts": 1, "max_fit_characters": 200_000, "max_train_tokens": 20_000, "max_validation_tokens": 5_000, "max_test_tokens": 5_000},
        "wikitext-103": {"max_fit_texts": 500, "max_fit_characters": None, "max_train_tokens": 20_000, "max_validation_tokens": 5_000, "max_test_tokens": 5_000},
        "one-billion-word": {"max_fit_texts": 500, "max_fit_characters": None, "max_train_tokens": 20_000, "max_validation_tokens": 5_000, "max_test_tokens": 5_000},
    },
    "medium": {
        "text8": {"max_fit_texts": None, "max_fit_characters": 1_000_000, "max_train_tokens": 200_000, "max_validation_tokens": 50_000, "max_test_tokens": 50_000},
        "enwik8": {"max_fit_texts": 1, "max_fit_characters": 1_000_000, "max_train_tokens": 200_000, "max_validation_tokens": 50_000, "max_test_tokens": 50_000},
        "wikitext-103": {"max_fit_texts": 2_000, "max_fit_characters": None, "max_train_tokens": 200_000, "max_validation_tokens": 50_000, "max_test_tokens": 50_000},
        "one-billion-word": {"max_fit_texts": 2_000, "max_fit_characters": None, "max_train_tokens": 200_000, "max_validation_tokens": 50_000, "max_test_tokens": 50_000},
    },
    "full": {
        "text8": {"max_fit_texts": None, "max_fit_characters": None, "max_train_tokens": None, "max_validation_tokens": None, "max_test_tokens": None},
        "enwik8": {"max_fit_texts": 1, "max_fit_characters": None, "max_train_tokens": None, "max_validation_tokens": None, "max_test_tokens": None},
        "wikitext-103": {"max_fit_texts": None, "max_fit_characters": None, "max_train_tokens": None, "max_validation_tokens": None, "max_test_tokens": None},
        "one-billion-word": {"max_fit_texts": None, "max_fit_characters": None, "max_train_tokens": None, "max_validation_tokens": None, "max_test_tokens": None},
    },
}

limits = PROFILES[PROFILE][DATASET_NAME].copy()
run_name = f"colab_{DATASET_NAME.replace('-', '_')}_{TOKENIZER_NAME}_{NGRAM_ORDER}gram_{PROFILE}_bpc"

load(DATASET_NAME)
print(f"Dataset ready: {DATASET_NAME}")
print({"profile": PROFILE, **limits})

config = NGramTrainingConfig(
    dataset_name=DATASET_NAME,
    tokenizer_name=TOKENIZER_NAME,
    order=NGRAM_ORDER,
    alpha=LAPLACE_ALPHA,
    max_vocab_size=None if TOKENIZER_NAME == "char" else 50_000,
    max_fit_texts=limits["max_fit_texts"],
    max_fit_characters=limits["max_fit_characters"],
    max_train_tokens=limits["max_train_tokens"],
    max_validation_tokens=limits["max_validation_tokens"],
    max_test_tokens=limits["max_test_tokens"],
    run_name=run_name,
)
config

In [ ]:
summary = train_ngram_language_model(
    config,
    prediction_contexts=PREDICTION_CONTEXTS,
    score_texts=SCORE_TEXTS,
    top_k=5,
)
summary["run_name"]

In [ ]:
metrics = load_metrics(summary["run_name"])
display(summarize_metrics(metrics))

timing_row = pd.DataFrame(
    [
        {
            "tokenizer": metrics["tokenizer"]["type"],
            "order": metrics["model"]["order"],
            "tokenizer_fit_s": round(metrics["timing"]["tokenizer_fit_seconds"], 4),
            "model_fit_s": round(metrics["timing"]["model_fit_seconds"], 4),
            "total_s": round(metrics["timing"]["total_seconds"], 4),
        }
    ]
)
display(timing_row)

In [ ]:
print("Prediction contexts")
for item in metrics["prediction_contexts"]:
    print(f"\nContext: {item['context_text']}")
    for pred in item["predictions"]:
        print(f"  {pred['token']!r}: {pred['probability']:.6f}")

print("\nScored texts")
for item in metrics["scored_texts"]:
    print(f"\nText: {item['text']}")
    print(f"  avg_nll = {item['average_negative_log_likelihood']:.4f}")
    print(f"  bpc     = {item['bits_per_character']:.4f}")
    print(f"  ppl     = {item['perplexity']:.4f}")

## Optional export

This cell zips the metrics JSON and tokenizer artifacts for the current run so you can download them from Colab.

In [ ]:
export_dir = PROJECT_ROOT / "colab_export_ngram_bpc"
zip_base = PROJECT_ROOT / "colab_export_ngram_bpc"

if export_dir.exists():
    shutil.rmtree(export_dir)

(export_dir / "metrics").mkdir(parents=True, exist_ok=True)
(export_dir / "artifacts").mkdir(parents=True, exist_ok=True)

run_name = metrics["run_name"]
shutil.copy2(METRICS_ROOT / f"{run_name}.json", export_dir / "metrics" / f"{run_name}.json")
shutil.copytree(ARTIFACT_ROOT / run_name, export_dir / "artifacts" / run_name, dirs_exist_ok=True)

zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=export_dir)
print("Created zip:", zip_path)

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(PROJECT_ROOT / "colab_export_ngram_bpc.zip"))
else:
    print(PROJECT_ROOT / "colab_export_ngram_bpc.zip")